<a href="https://colab.research.google.com/github/taloorlucas-eng/Finance-Mid-Term-Homework/blob/main/%E9%87%91%E8%9E%8D%E5%A4%A7%E4%BD%9C%E4%B8%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import matplotlib.font_manager as fm

#1.本文件除完成定价内容外，对保留利率树的复杂度进行了分析，并尝试增大N以验证。
#2.本文件进行了平价关系验证。


# 1. 参数设置区
R0 = 0.05                    # 初始连续复利零利率 (5%)
T = 5.0                      # 总期限 (年)
N_MAIN = 500

                # 总时间段数 (步数)(参数都可自行调节)
P_UP = 0.50                  # 风险中性测度下“向上”概率
STRIKE = 0.05                # 执行利率 / 互换固定利率
NOTIONAL = 1_000_000.0       # 名义本金
PAYMENT_INTERVAL = 1.0       # 付息间隔 (1.0表示每年)

MODEL_TO_PRICE = "BDT"       # 可选: "BDT" 或 "Ho-Lee"
PRODUCT_TO_PRICE = "cap"     # 可选: "cap", "floor", "swap"
SWAP_POSITION = "payer"      # 可选: 支付固定, 接收固定

# 模型波动率
SIGMA_DICT = {"BDT": 0.20, "Ho-Lee": 0.01}

# 市场初始贴现曲线 P(0,t)，默认平坦连续复利
MARKET_DISCOUNT_CURVE = lambda t: np.exp(-R0 * t)

# 2. 核心定价类：利率二叉树

class InterestRateTree:
    def __init__(self, model, N, p_up=0.5, calibrate=True):
        self.model = model.upper()
        self.N = N
        self.dt = T / N
        self.p_up = p_up
        self.q_down = 1.0 - p_up
        self.sigma = SIGMA_DICT[model]
        self.times = np.arange(N + 1) * self.dt

        # 市场贴现因子
        self.mkt_dfs = np.array([MARKET_DISCOUNT_CURVE(t) for t in self.times])

        # 记录每一层的中心利率水平
        self.levels = np.zeros(N)

        if calibrate:
            self.model_dfs = self._calibrate_curve()
        else:
            self.model_dfs = self.mkt_dfs # 简化处理不校准的情况

        self.fit_error = np.max(np.abs(self.model_dfs - self.mkt_dfs))

    def get_rates(self, step, level):
        """动态生成第 step 层的所有利率节点，降低空间复杂度"""
        moves = 2.0 * np.arange(step, -1, -1) - step
        shock = moves * self.sigma * np.sqrt(self.dt)
        if self.model == "HO-LEE":
            return level + shock
        else:
            return level * np.exp(shock)

    def _calibrate_curve(self):
        state_prices = np.array([1.0])
        fitted_dfs = np.zeros(self.N + 1)
        fitted_dfs[0] = 1.0

        for i in range(self.N):
            target_df = self.mkt_dfs[i + 1]
            moves = 2.0 * np.arange(i, -1, -1) - i
            shock = moves * self.sigma * np.sqrt(self.dt)

            if self.model == "HO-LEE":
                # Ho-Lee 解析解
                without_level = np.sum(state_prices * np.exp(-shock * self.dt))
                level_i = -np.log(target_df / without_level) / self.dt
            else:
                # BDT 二分法
                multipliers = np.exp(shock)
                low, high = 0.0, max(2.0 * R0, 1e-4)
                while np.sum(state_prices * np.exp(-high * multipliers * self.dt)) > target_df:
                    high *= 2.0
                for _ in range(50):
                    mid = (low + high) / 2.0
                    if np.sum(state_prices * np.exp(-mid * multipliers * self.dt)) > target_df:
                        low = mid
                    else:
                        high = mid
                level_i = (low + high) / 2.0

            self.levels[i] = level_i
            rates = self.get_rates(i, level_i)

            # 更新状态价格并折现
            discounted_sp = state_prices * np.exp(-rates * self.dt)
            fitted_dfs[i + 1] = np.sum(discounted_sp)

            # 向下传播到下一层
            next_sp = np.zeros(i + 2)
            next_sp[:-1] += self.p_up * discounted_sp
            next_sp[1:] += self.q_down * discounted_sp
            state_prices = next_sp

        return fitted_dfs

    def _zcb_price(self, start_step, end_step):
        bond_vals = np.ones(end_step + 1)
        for k in range(end_step - 1, start_step - 1, -1):
            rates = self.get_rates(k, self.levels[k])
            expected = self.p_up * bond_vals[:-1] + self.q_down * bond_vals[1:]
            bond_vals = np.exp(-rates * self.dt) * expected
        return bond_vals

    def price_derivatives(self, strike, notional, pay_interval):
        """同时对 Cap, Floor, Swap 进行定价"""
        steps_per_pay = int(round(pay_interval / self.dt))
        prices = {"cap": 0.0, "floor": 0.0, "swap": 0.0}
        state_prices = np.array([1.0])

        for i in range(self.N):
            if i % steps_per_pay == 0:
                bond_prices = self._zcb_price(i, i + steps_per_pay)
                # 浮动 - 固定
                float_minus_fixed = 1.0 - (1.0 + strike * pay_interval) * bond_prices

                prices["cap"] += np.sum(state_prices * notional * np.maximum(float_minus_fixed, 0))
                prices["floor"] += np.sum(state_prices * notional * np.maximum(-float_minus_fixed, 0))
                prices["swap"] += np.sum(state_prices * notional * float_minus_fixed)

            rates = self.get_rates(i, self.levels[i])
            discounted_sp = state_prices * np.exp(-rates * self.dt)
            next_sp = np.zeros(i + 2)
            next_sp[:-1] += self.p_up * discounted_sp
            next_sp[1:] += self.q_down * discounted_sp
            state_prices = next_sp

        return prices


# 3. 主程序：
if __name__ == "__main__":


    # 建立定价树与计算
    t_start = time.perf_counter()
    pricer = InterestRateTree(MODEL_TO_PRICE, N_MAIN, p_up=P_UP)
    prices = pricer.price_derivatives(STRIKE, NOTIONAL, PAYMENT_INTERVAL)

    # 互换买卖方调整
    if SWAP_POSITION == "receiver":
        prices["swap"] = -prices["swap"]

    t_end = time.perf_counter()

    # 打印参数信息
    print("\n【模型基础参数】")
    print(f"选择模型       : {MODEL_TO_PRICE}")
    print(f"时间步数 (N)   : {N_MAIN}")
    print(f"向上风险中性概率 (p)   : {P_UP:.2f}")
    print(f"执行利率       : {STRIKE:.2%}")
    print(f"名义本金       : {NOTIONAL:,.2f}")
    print(f"最大曲线拟合误差: {pricer.fit_error:.3e}")

    # 打印价格信息
    print("\n【零时刻 (t=0) 定价结果】")
    print(f"利率上限 (Cap) 价格 : {prices['cap']:,.2f}")
    print(f"利率下限 (Floor) 价格: {prices['floor']:,.2f}")
    print(f"利率互换 (Swap) 价格 : {prices['swap']:,.2f} ({SWAP_POSITION} 视角)")

    # 验证平价关系: Cap - Floor = Payer Swap
    parity_diff = prices["cap"] - prices["floor"] - (prices["swap"] if SWAP_POSITION=="payer" else -prices["swap"])
    print("\n【Cap-Floor-Swap 平价关系验证】")
    print(f"误差绝对值 : {abs(parity_diff):.3e} (这里合理认为极小值即代表满足平价关系)")

    print(f"\n【计算效率】")
    print(f"建树与校准+定价总耗时 : {t_end - t_start:.4f} 秒")
    # 5. 可选部分的完成：提升计算效率，挑战极大 N 值,以cap为例
    print("1. 空间复杂度优化：摒弃传统保存整棵二叉树")
    print("   采用动态生成单层切片的方式，将空间复杂度压缩至 O(N)。")
    print("2. 时间复杂度优化：经过分析，复杂度仍为O（N^2)")
    print("   但降低了实际运行时间常数。")

    N_EXmwd = 2000  # 设置一个比较大的N尝试
    print(f"\n>>> 大规模二叉树 N = {N_EXmwd} ...")

    t_opt_start = time.perf_counter()
    pricer_ext = InterestRateTree(MODEL_TO_PRICE, N_EXmwd, p_up=P_UP, calibrate=True)
    prices_ext = pricer_ext.price_derivatives(STRIKE, NOTIONAL, PAYMENT_INTERVAL)
    t_opt_end = time.perf_counter()

    print(f" N={N_EXmwd} 步下的 Cap 定价为 : {prices_ext['cap']:,.4f}")
    print(f"用时 : {t_opt_end - t_opt_start:.4f} 秒")






【模型基础参数】
选择模型       : BDT
时间步数 (N)   : 500
向上风险中性概率 (p)   : 0.50
执行利率       : 5.00%
名义本金       : 1,000,000.00
最大曲线拟合误差: 2.220e-16

【零时刻 (t=0) 定价结果】
利率上限 (Cap) 价格 : 24,344.22
利率下限 (Floor) 价格: 18,860.32
利率互换 (Swap) 价格 : 5,483.90 (payer 视角)

【Cap-Floor-Swap 平价关系验证】
误差绝对值 : 1.819e-12 (这里合理认为极小值即代表满足平价关系)

【计算效率】
建树与校准+定价总耗时 : 0.4272 秒
1. 空间复杂度优化：摒弃传统保存整棵二叉树
   采用动态生成单层切片的方式，将空间复杂度压缩至 O(N)。
2. 时间复杂度优化：经过分析，复杂度仍为O（N^2)
   但降低了实际运行时间常数。

>>> 大规模二叉树 N = 2000 ...
 N=2000 步下的 Cap 定价为 : 24,340.1365
用时 : 3.0635 秒
